# Phase 5 Pipeline B — Full recoloration of treated pairs

Processes the 18 treated pairs, generating up to 15 approved recolored images per pair (with up to 25 attempts each to compensate for VLM rejections). Total: ~270 approved images expected.

**Runtime:** ~2-3h on A100. The bottleneck is the VLM judge — SAM2 + Grounding DINO segmentation is fast, but Qwen2.5-VL is loaded in addition and takes ~5-8s per image.

**Idempotent:** if Colab disconnects, just rerun; it picks up where it left off.

## 1. Mount Drive and clone repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REPO_URL = "https://github.com/GabrielmAlves/dgm-2026.1.git"
BRANCH = "feature/generate_datasets"

import os, subprocess
REPO_DIR = "/content/binding-research"
if os.path.exists(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
%cd $REPO_DIR/projects/challenges-in-attribute-control

## 2. Install deps (SAM2, transformers — usually fast on rerun)

In [ ]:
!pip install -q uv
!uv pip install -e . --system
!pip install -q git+https://github.com/facebookresearch/sam2.git
!pip install -q transformers accelerate

## 3. HF login

In [ ]:
from huggingface_hub import login
login()

## 4. Confirm GPU (need A100 for VLM + SAM together)

In [ ]:
import torch
assert torch.cuda.is_available()
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU: {torch.cuda.get_device_name(0)} ({vram:.1f} GB)')
assert vram >= 35, 'A100 strongly recommended (SAM2+DINO ~10GB + Qwen-VL ~16GB)'

## 5. Rewrite the source_pool paths to Drive locations

Same trick as the smoke notebook — the pool CSV has Windows paths and we redirect them to where the PNGs actually live on Drive.

In [ ]:
import csv
from pathlib import Path

POOL_DRIVE = Path('/content/drive/MyDrive/binding-research/exp5_pool/source_pool.csv')
DRIVE_ROOT = Path('/content/drive/MyDrive/binding-research/finetuning')

out_csv = Path('/content/source_pool_colab.csv')
with POOL_DRIVE.open(newline='') as fin, out_csv.open('w', newline='') as fout:
    reader = csv.DictReader(fin)
    writer = csv.DictWriter(fout, fieldnames=reader.fieldnames)
    writer.writeheader()
    for row in reader:
        if row['source_collection'] == 'control':
            root = DRIVE_ROOT / 'control_candidates'
        else:
            root = DRIVE_ROOT / 'canonical_extra_candidates'
        win_path = row['path'].replace('\\', '/')
        parts = win_path.split('/')
        rel = '/'.join(parts[-3:])
        row['path'] = str(root / rel)
        writer.writerow(row)
print(f'Wrote pool CSV: {out_csv}')

## 6. Upload split CSV to Colab (one-off)

The script needs `finetuning_split.csv` to know which pairs are 'treated'. If you haven't put it in Drive yet, upload `results/exp5_split/finetuning_split.csv` from your Windows machine to `MyDrive/binding-research/exp5_split/` first.

In [ ]:
import os
SPLIT_DRIVE = '/content/drive/MyDrive/binding-research/exp5_split/finetuning_split.csv'
assert os.path.exists(SPLIT_DRIVE), f'Upload split to {SPLIT_DRIVE}'
os.makedirs('results/exp5_split', exist_ok=True)
if os.path.lexists('results/exp5_split/finetuning_split.csv'):
    os.remove('results/exp5_split/finetuning_split.csv')
os.symlink(SPLIT_DRIVE, 'results/exp5_split/finetuning_split.csv')
print('Split CSV linked')

## 7. Run the full recoloration

Output goes to `/content/recolor_treated/`. This is where Colab disconnects would hurt — but the script is idempotent so just rerun this cell to resume.

In [ ]:
!python experiments/exp5_recolor_full.py \
    --pool /content/source_pool_colab.csv \
    --split results/exp5_split/finetuning_split.csv \
    --out-root /content/recolor_treated \
    --config configs/judge_default.yaml \
    --per-pair 15 \
    --max-attempts 25

## 8. Quick inspection of results

In [ ]:
import pandas as pd
import json

appr = pd.read_csv('/content/recolor_treated/approved.csv')
rej  = pd.read_csv('/content/recolor_treated/rejected.csv')
print(f'Approved: {len(appr)}')
print(f'Rejected: {len(rej)}')
print(f'Acceptance rate: {len(appr)/(len(appr)+len(rej)):.1%}')
print()
print('=== Approved per pair ===')
print(appr.groupby(['object','target_color']).size().to_string())
print()
print('=== Rejection by stage ===')
print(rej.groupby(['stage']).size().to_string())
print()
with open('/content/recolor_treated/summary.json') as f:
    summary = json.load(f)
print(f'Total approved: {summary["total_approved"]}')

## 9. Save to Drive

In [ ]:
import shutil
DEST = '/content/drive/MyDrive/binding-research/finetuning/recolor_treated'
shutil.copytree('/content/recolor_treated', DEST, dirs_exist_ok=True)
print(f'Saved to {DEST}')